# QA Retrieval Chatbot (RAG): BrewBean Coffee Co.

**Business (dummy data):** *BrewBean Coffee Co.* is a fictional coffee chain with 12 cafes in Lahore, Karachi and Islamabad.
Its company handbook (`data/brewbean_company_handbook.pdf`) has the menu and prices, refund and delivery policies, the BeanPoints
loyalty program and FAQs.

**Goal:** Build a chatbot that answers customer questions **only from BrewBean's own documents**.

ChatGPT doesn't know about our company, so we use **RAG (Retrieval-Augmented Generation)**: we first store our documents in a
vector database, then for each question we retrieve the relevant parts and give them to the LLM so it can answer from our data.

#### How many phases does a QA Retrieval chatbot have?
#### A RAG system has two main phases:
1️⃣ **Insertion (Indexing)**: prepare the knowledge (done once)

2️⃣ **Retrieval (Answering)**: answer a question (every time)

```
1. INDEXING (once)
   PDF ──► Document Loader ──► Text Splitter (chunks) ──► Embedding Model ──► Vector DB (Pinecone)

2. RETRIEVAL (every question)
   User question ──► Retriever (embed + similarity search) ──► Relevant chunks
                                                                    │
                           Prompt (instructions + context + question) ──► LLM ──► Final answer
```

### Components of a RAG system (architecture)
*Architecture = a system's overall structure, its parts, and how those parts communicate.*

| # | Component | Job | In this notebook |
|---|---|---|---|
| 1 | Documents | Knowledge source | `brewbean_company_handbook.pdf` |
| 2 | Document loader | Reads the files | `PyMuPDFLoader` |
| 3 | Text splitter | Makes chunks | `RecursiveCharacterTextSplitter` |
| 4 | Embedding model | Turns text into numbers | OpenAI `text-embedding-3-small` |
| 5 | Vector database | Stores the vectors | Pinecone |
| 6 | Retriever | Finds the relevant chunks | `vector_store.as_retriever()` |
| 7 | LLM | Writes the answer | OpenAI `gpt-4.1-mini` |
| 8 | User interface | Where users chat | Notebook cells / chat loop |

# How many types of databases are there?
1. Relational databases (PostgreSQL, MySQL)
2. Non-relational databases (MongoDB)
3. Vector databases (Pinecone, Qdrant, ChromaDB, FAISS)
    1. Cloud-based, online: Pinecone, Qdrant (live 24/7, accessible from anywhere)
    2. Local: ChromaDB, FAISS (run on your own laptop)

**Normal DB vs vector DB:** a normal database finds *exact matches* (`name = "Cappuccino"`). A vector database finds *similar meaning*:
the query "milky coffee with foam" finds the nearest vectors (Cappuccino, Latte) even though no keyword matches.

### Insertion Phase
📄 Add documents
(PDFs, text files, notes, websites)

✂️ Split the documents into small chunks
(So the AI can understand them better)

🔢 Convert the text into embeddings (numbers)
(Using an embedding model)

🗄️ Store the embeddings in a vector database
(Like FAISS, Pinecone, Chroma)

### Retrieval phase

❓ User asks a question

🔢 Convert the question into an embedding

🔍 Search for similar embeddings in the vector database

📄 Retrieve the most relevant text chunks

🤖 Send the retrieved text + question to the LLM

💬 The AI generates the final answer

### Setup: API keys from the `.env` file
API keys are **never hard-coded** in the notebook. Create a `.env` file in the project folder (copy `.env.example`):

```
OPENAI_API_KEY=your-openai-key
PINECONE_API_KEY=your-pinecone-key
```

`.env` is listed in `.gitignore`, so it never goes to GitHub. `load_dotenv()` reads the keys into environment variables.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    print(key, "loaded" if os.getenv(key) else "MISSING: add it to your .env file")

---
# Phase 1: Indexing
### Step 1: Load the document
We load the PDF first, and then we build the chatbot on top of it.
`PyMuPDFLoader` reads the PDF and returns a **list of `Document` objects**, one per page.
Each `Document` has `page_content` (the text) and `metadata` (source, page number, total pages...).

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "data/brewbean_company_handbook.pdf"
loader = PyMuPDFLoader(file_path)
docs = loader.load()
docs[0]

In [ ]:
# Python lists start at index 0, so docs[1] is the second page
docs[1].page_content

In [ ]:
docs[1].metadata

In [ ]:
len(docs)  # number of pages in the PDF

### Step 2: Split into chunks
We split the pages into small **chunks** because:
- every embedding model has a limit on how much text it can convert at once
- small focused chunks give more accurate search results, and we send the LLM only what it needs

**`chunk_size=1000`**: each chunk is at most 1000 characters.
**`chunk_overlap=200`**: each new chunk repeats the last 200 characters of the previous one, so a sentence cut at a boundary
doesn't lose its context.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = splitter.split_documents(docs)
print(len(splits))

In [ ]:
# Look at one chunk and its metadata
print(splits[2].page_content)
print(splits[2].metadata["page"])

### Step 3: Embedding model (text → numbers)
**Embeddings turn the meaning of text into numbers (vectors).** Texts with similar meaning get similar numbers:

| Text | Vector (example values) |
|---|---|
| "How much is a cappuccino?" | `[0.82, 0.15, -0.31, …]` |
| "What does a cappuccino cost?" | `[0.79, 0.18, -0.28, …]` |

So the customer's question "price of your Spanish latte" can match the menu line "Spanish Latte: PKR 900" even with different words.
*Embeddings don't generate answers. They make similarity search possible.*

`dimensions=1024` is the length of each vector. The Pinecone index **must use the same dimension**.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024
)

In [ ]:
# Test: is the embedding model working? Convert a sentence into numbers
vector = embedding.embed_query("Hello, I am a BrewBean customer")
print("Dimensions:", len(vector))
print(vector[:5])

### Step 4: Prepare the vector database (Pinecone)
Now we need a vector database account where we will store these vectors. Follow these steps:
1. Go to [pinecone.io](https://www.pinecone.io/), create an account and log in
2. Create an API key and paste it into the `.env` file as `PINECONE_API_KEY`

An **index** is like a folder in Pinecone where our vectors are stored.

We put all the Pinecone work in one **class**, `PineconeDB`, so we can call `create_index`, `delete_index`, `insert` and `retrieve`
as methods on one object.

In [ ]:
import os
import hashlib
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore


class PineconeDB:
    def __init__(self, api_key: str | None = None):
        self.pc = Pinecone(api_key=api_key or os.getenv("PINECONE_API_KEY"))

    def create_index(self, index_name: str, dimension: int, metric: str = "cosine") -> str:
        if self.pc.has_index(index_name):
            return f"Index '{index_name}' already exists."
        self.pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
        return f"Index '{index_name}' created."

    def delete_index(self, index_name: str) -> str:
        if not self.pc.has_index(index_name):
            return f"Index '{index_name}' does not exist."
        self.pc.delete_index(index_name)
        return f"Index '{index_name}' deleted."

    def _store(self, index_name: str, embedding, namespace: str | None = None) -> PineconeVectorStore:
        return PineconeVectorStore(index=self.pc.Index(index_name), embedding=embedding, namespace=namespace)

    def insert(self, index_name: str, documents, embedding, namespace: str | None = None) -> list[str]:
        # same chunk -> same id, so running this again overwrites instead of duplicating
        ids = [hashlib.md5(f"{d.metadata.get('source')}|{d.metadata.get('page')}|{d.page_content}".encode()).hexdigest() for d in documents]
        return self._store(index_name, embedding, namespace).add_documents(documents, ids=ids)

    def retrieve(self, index_name: str, query: str, embedding, k: int = 4, namespace: str | None = None):
        return self._store(index_name, embedding, namespace).similarity_search(query, k=k)


pinecone_db = PineconeDB()

#### Create the index

In [ ]:
index_name = "brewbean"
print(pinecone_db.create_index(index_name, dimension=1024))  # must match the embedding dimensions

#### Insert the chunks
Each chunk is converted into a vector and stored in Pinecone along with its text and metadata.
After this, refresh the index in the Pinecone dashboard and you can see the records.

In [ ]:
ids = pinecone_db.insert(index_name, splits, embedding)
print(f"Inserted {len(ids)} chunks into '{index_name}'")
print(pinecone_db.pc.Index(index_name).describe_index_stats())

#### Retrieve (check that the data was stored)
A quick retrieval check. **No LLM is needed here**, only the embedding model: the question is converted into a vector and
Pinecone returns the `k` most similar chunks.

In [ ]:
results = pinecone_db.retrieve(index_name, "What is the refund policy?", embedding, k=3)
for doc in results:
    print(doc.metadata.get("page"), "|", doc.page_content[:200])
    print("-" * 80)

In [ ]:
len(results)

In [ ]:
# Run only when you want to drop the whole index
# print(pinecone_db.delete_index(index_name))

---
# Phase 2: Retrieval (Answering)
**Retrieval** is the *process* of finding only the relevant information in a large knowledge base.
The **retriever** is the *component* that does it, like a librarian who brings you only the books you asked for, not all 10,000.

Retrieval strategies: **similarity search** (by meaning, used here), **metadata filtering** (by document/category/date) and
**hybrid search** (keywords + meaning).

Simple RAG flow, step by step:

1. ❓ User asks a question
2. 🔍 The retriever converts the question into an embedding and finds the top-k similar chunks in Pinecone
3. 📄 Join the retrieved chunks into one **context** text
4. 📝 Put the context + question into a **prompt**
5. 🤖 Send the prompt to the LLM → 💬 final answer

First we do each step manually, then we join all the steps into one chain.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_pinecone import PineconeVectorStore

# LLM that will write the final answer (reads OPENAI_API_KEY from .env)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Retriever: connects to the SAME Pinecone index + SAME embedding model we used in the Indexing phase
vector_store = PineconeVectorStore(index=pinecone_db.pc.Index(index_name), embedding=embedding)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # k = how many chunks to bring back

#### Steps 1 & 2: User question → retrieve the relevant chunks
The retriever turns the question into an embedding and returns the top-k most similar chunks from Pinecone.

In [ ]:
question = "How much does a Spanish Latte cost and is there a student discount?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"Chunk {i} | page {int(doc.metadata.get('page', 0))}")
    print(doc.page_content[:300])
    print("-" * 80)

#### Step 3: Join the retrieved chunks into one context
The retrieved data has embeddings, metadata and page content. We only need the text, so we join all the chunks into one context block.

In [ ]:
def format_docs(docs):
    """Join all retrieved chunks into one big text block."""
    return "\n\n".join(doc.page_content for doc in docs)

context = format_docs(retrieved_docs)
print(context)

#### Step 4: Put the context + question into a prompt
The LLM receives three things: **instructions** (persona + rules), the **retrieved context** and the **user question**.
We tell the LLM to answer **only** from the context, so it doesn't make things up.

⚠️ The keys `{context}` and `{question}` in the template must match the dictionary keys we pass to `invoke`. Otherwise you get an error.

In [ ]:
prompt = ChatPromptTemplate.from_template("""You are a helpful customer support assistant for BrewBean Coffee Co.
Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
""")

# See the exact prompt that will go to the LLM
final_prompt = prompt.invoke({"context": context, "question": question})
print(final_prompt.to_string())

#### Step 5: Send the prompt to the LLM → final answer
*The retriever finds the information. The LLM turns it into a clear answer.*

In [ ]:
response = llm.invoke(final_prompt)
print(response.content)

#### Full RAG pipeline in one chain
The same 5 steps, joined with `|` (LCEL, LangChain Expression Language):

`question → retriever → format_docs → prompt → llm → answer text`

- `retriever | format_docs` fills `{context}`
- `RunnablePassthrough()` passes the user question as-is into `{question}`
- `StrOutputParser()` turns the AI message into plain text

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()   # AIMessage -> plain text
)

In [ ]:
print("Q1:", rag_chain.invoke("Can I return coffee beans I ordered online?"), "\n")
print("Q2:", rag_chain.invoke("How many BeanPoints do I need for a free drink?"), "\n")
print("Q3:", rag_chain.invoke("Is delivery free?"), "\n")
print("Q4:", rag_chain.invoke("Who won the cricket world cup?"))  # not in the document -> "I don't know"

#### Bonus: a simple chat interface
Type your questions. Type `exit` to stop.

In [ ]:
while True:
    user_q = input("You: ")
    if user_q.strip().lower() in {"exit", "quit"}:
        break
    print("BrewBean Bot:", rag_chain.invoke(user_q), "\n")

---
## Summary: the complete RAG pipeline

| Phase | Step | Tool used |
|---|---|---|
| Indexing (once) | 1. Load the business PDF | `PyMuPDFLoader` |
| | 2. Split into chunks (1000 / overlap 200) | `RecursiveCharacterTextSplitter` |
| | 3. Convert to embeddings (1024 dims) | `OpenAIEmbeddings` |
| | 4. Store vectors + text + metadata | Pinecone (`PineconeDB` class) |
| Retrieval (every question) | 5. Find the top-3 relevant chunks | `vector_store.as_retriever()` |
| | 6. Build the prompt (instructions + context + question) | `ChatPromptTemplate` |
| | 7. Generate the answer from the context | `ChatOpenAI` + LCEL chain |

**Real-world uses:** hospitals (policies), universities (admissions, fees), e-commerce (products, returns), companies (HR docs).

**Security:** all API keys are loaded from `.env`, which is excluded by `.gitignore`. No secrets are in this notebook or the repository.